In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, skew, kurtosis
from google.cloud import bigquery
from google.oauth2 import service_account
# ====================================================================
# PHẦN 1: KẾT NỐI VÀ HÚT DATA TỪ BIGQUERY
# ====================================================================
project_id = "jda-k1"
dataset_id = "cuongnm_data_pipeline"
key_path = r"C:\Users\DELL\Desktop\Project\Project Market risk\Key - path 2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

print("⏳ Đang cắm ống hút dữ liệu vào BigQuery...")
# Viết lệnh SQL để kéo Data sạch về (Sắp xếp theo ngày tăng dần để tính Return chuẩn)
query = f"""
    SELECT Date, EURUSD, VANG, HPG, VNM, TCB, VN30F1M
    FROM jda-k1.cuongnm_data_pipeline.Market_Risk_Data
    ORDER BY Date ASC
"""
df_gia = client.query(query).to_dataframe()

# Xử lý Index cho DataFrame
df_gia.set_index('Date', inplace=True)
df_gia.index = pd.to_datetime(df_gia.index)

print("✅ Đã kéo thành công dữ liệu từ BigQuery!")
# ====================================================================
# PHẦN 2: LÕI TOÁN HỌC 
# ====================================================================
class MarketRisk:
    def __init__(self, data_frame, weights_base, von_ban_dau):
        self.von = von_ban_dau
        # Ép thứ tự cột chuẩn để Toán học không bị nhân nhầm tài sản
        df_gia = data_frame[['EURUSD', 'VANG', 'HPG', 'VNM', 'TCB', 'VN30F1M']]
        df_returns = df_gia.pct_change().dropna()
        self.last_prices = df_gia.iloc[-1] # Lấy giá ngày cuối cùng để lát nữa quy đổi ra Hợp đồng
        self.weights_base = np.array(weights_base)
        
        # Bóc tách 3 mã Cổ phiếu (HPG, VNM, TCB nằm ở vị trí số 2, 3, 4 trong mảng)
        w_hpg, w_vnm, w_tcb = self.weights_base[2], self.weights_base[3], self.weights_base[4]
        self.w_equity_total = w_hpg + w_vnm + w_tcb
        
        # Tính Beta và Tỷ trọng Phái sinh tự động
        if self.w_equity_total != 0:
            # Lợi suất trung bình của rổ Cổ phiếu
            eq_returns = (df_returns['HPG']*w_hpg + df_returns['VNM']*w_vnm + df_returns['TCB']*w_tcb) / self.w_equity_total
            # Tính Hiệp phương sai (Covariance) và Phương sai (Variance)
            cov_matrix = np.cov(eq_returns, df_returns['VN30F1M'])
            self.beta = cov_matrix[0, 1] / cov_matrix[1, 1]
        else:
            self.beta = 0.0
            
        # Tỷ trọng Hedge = -(Beta * Tỷ trọng Cổ phiếu)
        self.w_f1m = -self.beta * self.w_equity_total
        
        # Ghép thành 6 tỷ trọng và nhân với ma trận 6 cột
        self.weights_full = np.append(self.weights_base, self.w_f1m)
        self.port_returns = df_returns.dot(self.weights_full)
    def get_hedge_contracts(self):
        """Quy đổi Tỷ trọng Phái sinh ra Số lượng Hợp đồng thực tế"""
        import math
        gia_f1m = self.last_prices['VN30F1M']
        gia_tri_hedge = self.w_f1m * self.von
        
        # Tính số HĐ (chia cho Hệ số nhân 100,000 VND/điểm)
        so_hd = gia_tri_hedge / (gia_f1m * 100000)
        return math.ceil(abs(so_hd)) if so_hd < 0 else math.floor(so_hd)
    #Concentration_risk
    def check_concentration_risk(self, hhi_threshold=0.25, single_asset_limit=0.30):
        """Đo lường Rủi ro tập trung (HHI) và Hạn mức tỷ trọng"""
        # Tính HHI (tổng bình phương các tỷ trọng đầy đủ bao gồm cả lệnh Hedge)
        hhi_index = np.sum(self.weights_full ** 2)
        
        assets = ['EURUSD', 'VANG', 'HPG', 'VNM', 'TCB', 'VN30F1M']
        
        print(f"\n[3] KIỂM SOÁT RỦI RO TẬP TRUNG (CONCENTRATION RISK):")
        print(f"  - Chỉ số HHI hiện tại: {hhi_index:.4f}")
        
        if hhi_index > hhi_threshold:
            print(f"  -> 🔴 [RED FLAG] HHI vượt ngưỡng an toàn ({hhi_threshold}). Danh mục quá cô đặc!")
        else:
            print(f"  -> 🟢 [SAFE] HHI nằm trong ngưỡng an toàn.")
            
        print(f"  - Cảnh báo Hạn mức tài sản đơn lẻ (>{single_asset_limit * 100:.0f}%):")
        breach_count = 0
        for asset, weight in zip(assets, self.weights_full):
            # Dùng abs() vì lệnh Short phái sinh mang tỷ trọng âm, nhưng rủi ro quy mô là trị tuyệt đối
            if abs(weight) > single_asset_limit:
                print(f"     !!! Mã {asset} chiếm {abs(weight)*100:.2f}% (Vượt mức {single_asset_limit*100:.0f}%)")
                breach_count += 1
                
        if breach_count == 0:
            print("     ✅ Không có tài sản nào vi phạm hạn mức tối đa.")
    #VaR
    def historical_var(self, confidence_level=0.99):
        """Historical VaR: Rủi ro dựa trên lịch sử thực tế"""
        alpha_percent = (1 - confidence_level) * 100
        h_var_pct = np.percentile(self.port_returns, alpha_percent)
        return h_var_pct * self.von

    def parametric_var(self, confidence_level=0.99):
        """Parametric VaR: Rủi ro giả định theo phân phối chuẩn (Hình chuông)"""
        mu = self.port_returns.mean()
        sigma = np.std(self.port_returns)
        z_score = norm.ppf(1 - confidence_level)
        return (mu + z_score * sigma) * self.von
        
    def cornish_fisher_var(self, confidence_level=0.99):
        """Cornish-Fisher VaR: Nâng cấp của Parametric, trị được Đuôi Béo"""
        mu = self.port_returns.mean()
        sigma = np.std(self.port_returns)
        S = skew(self.port_returns)
        K = kurtosis(self.port_returns, fisher=False)
        z = norm.ppf(1 - confidence_level)
        
        # Công thức uốn cong Z-score để bọc lót rủi ro Thiên nga đen (FRM)
        z_cf = z + (z**2 - 1)*S/6 + (z**3 - 3*z)*(K-3)/24 - (2*z**3 - 5*z)*(S**2)/36
        return (mu + z_cf * sigma) * self.von
    #Expected_Shortfall
    def frtb_10day_expected_shortfall(self, confidence_level=0.975):
        """
        Chuẩn FRTB: 10-day ES ở độ tin cậy 97.5%.
        Sử dụng Overlapping 10-day returns để tránh giả định Phân phối chuẩn sai lệch.
        """
        returns_10d = ((1+ self.port_returns).rolling(window=10).apply(np.prod, raw=True) - 1).dropna()
        var_10d_pct = np.percentile(returns_10d, ((1 - confidence_level) * 100))
        tail_events = returns_10d[returns_10d <= var_10d_pct]
        return tail_events.mean() * self.von
    def stressed_expected_shortfall(self, confidence_level=0.975, window=250):
        """
        Chuẩn FRTB: Stressed ES.
        Trượt cửa sổ 250 ngày trên toàn bộ lịch sử để tìm ra giai đoạn khủng hoảng
        cực đoan nhất đối với danh mục hiện tại (Ví dụ: Đại dịch Covid 2020).
        """
        max_es_loss = 0
        # Lăn cửa sổ 250 ngày qua toàn bộ tập dữ liệu
        for i in range(len(self.port_returns) - window + 1):
            period_returns = self.port_returns.iloc[i : i + window]
            
            var_str_es = np.percentile(period_returns, (1 - confidence_level) * 100)
            es = period_returns[period_returns <= var_str_es].mean() * self.von
            
            # Tìm mức lỗ sâu nhất 
            if es < max_es_loss:
                max_es_loss = es
                
        return max_es_loss
    def expected_shortfall(self, confidence_level=0.99):
        """Expected Shortfall (CVaR): Mức lỗ trung bình của những ngày thảm họa"""
        h_var_pct = self.historical_var(confidence_level) / self.von
        tail_events = self.port_returns[self.port_returns <= h_var_pct]
        return tail_events.mean() * self.von
    
    #Backtesting theo kiểu này bị lỗi in-of-sample
    # def backtest(self, confidence_level=0.99):
    #     """Backtesting: Đếm số lần thực tế lỗ nặng hơn VaR (Chuẩn Basel)"""
    #     h_var_pct = self.historical_var(confidence_level) / self.von
    #     exceptions = self.port_returns[self.port_returns < h_var_pct]
    #     return len(exceptions)
    #Backtesting Chuẩn Basel (Out-of-Sample)
    def backtest(self, confidence_level=0.99, window=250):
        """
        Backtesting Out-of-sample (Chuẩn Basel III / FRTB): 
        Dùng cửa sổ trượt 250 ngày (1 năm) để tính VaR ngày T-1, 
        rồi so sánh với PnL thực tế của ngày T.
        """
        alpha_percent = (1 - confidence_level) * 100
        
        # 1. TÍNH VAR TRƯỢT & SHIFT 1 NGÀY (Vectorized)
        rolling_var_pct = self.port_returns.rolling(window=window).apply(lambda x: np.percentile(x, alpha_percent), raw=True)
        # Phải đẩy đường VaR lùi lại 1 ngày, để lấy VaR chốt phiên hôm qua đối chiếu với Return hôm nay.
        var_limit_shifted = rolling_var_pct.shift(1)
        
        # 2. Lấy dữ liệu: Bỏ qua 250 ngày đầu tiên vì chưa đủ dữ liệu mồi
        test_returns = self.port_returns[window:]
        test_limits = var_limit_shifted[window:]
        
        # 4. ĐÓNG GÓI THÀNH DATAFRAME LỊCH SỬ và cắm cờ những ngày đâm thủng VaR
        df_history = pd.DataFrame({
            "Date": test_returns.index,
            "Actual_PnL_VND": test_returns.values * self.von,
            "VaR_Limit_VND": test_limits.values * self.von
        })
        df_history['Is_Breach'] = (df_history['Actual_PnL_VND'] < df_history['VaR_Limit_VND']).astype(int)
        
        so_lan_vi_pham = df_history['Is_Breach'].sum()
        tong_so_ngay_test = len(df_history)
        
        return so_lan_vi_pham, tong_so_ngay_test, df_history

# ====================================================================
# Đưa dữ liệu vào
# ====================================================================
if __name__ == "__main__":
    
    # 1. Cấu hình danh mục: Điền Vốn đầu tư, Nhập Độ tin cậy (CI), nhập Tỷ trọng danh mục
    # Thứ tự cột trong data_set.csv là: EURUSD, VANG, HPG, VNM, TCB và công cụ hedge VN30F1M
    print("="*75)
    print(" HỆ THỐNG KHỞI TẠO THÔNG SỐ RỦI RO (DESK INPUT) ".center(75))
    print("="*75)
    # while True:
    #     try:
    #         von_nhap = input("💰 Nhập quy mô Vốn tự có (VND) [VD: 10000000000]: ")
    #         von_dau_tu = float(von_nhap.replace(',', '')) # Cho phép nhập dấu phẩy
    #         if von_dau_tu <= 0:
    #             print("❌ Vốn phải lớn hơn 0. Ngân hàng không cấp vốn âm. Nhập lại!")
    #             continue
    #         break
    #     except ValueError:
    #         print("❌ Lỗi định dạng! Vui lòng chỉ nhập con số.")

    # while True:
    #     try:
    #         CI = float(input("🎯 Nhập Độ tin cậy VaR/ES (0 < CI < 1) [VD: 0.99]: "))
    #         if not (0 < CI < 1):
    #             print("❌ Confidence Interval phải nằm trong khoảng (0, 1). Nhập lại!")
    #             continue
    #         break
    #     except ValueError:
    #         print("❌ Lỗi định dạng! Vui lòng nhập số thập phân.")

    # while True:
    #     try:
    #         print("\n📊 Danh sách tài sản theo thứ tự: [EURUSD, VANG, HPG, VNM, TCB và công cụ hedge VN30F1M]")
    #         str_ty_trong = input("⚖️ Nhập tỷ trọng cách nhau bằng dấu phẩy [VD: 0.2, 0.4, 0.15, 0.1, 0.2]: ")
            
    #         # Tách chuỗi và ép kiểu sang số thực
    #         ty_trong = [float(x.strip()) for x in str_ty_trong.split(',')]
            
    #         # Kiểm tra xem có khớp số lượng tài sản không (ở đây là 4)
    #         if len(ty_trong) != 5:
    #             print(f"❌ Danh mục có 5 tài sản, nhưng cậu vừa nhập {len(ty_trong)} tỷ trọng. Nhập lại!")
    #             continue

    
    #         # Lưu ý nghiệp vụ: Tổng tỷ trọng có thể > 1 (hoặc < 0) do đòn bẩy phái sinh (Margin)
    #         # Khối Risk không cấm điều này, nhưng ghi nhận lại rủi ro đòn bẩy.
    #         tong_leverage = sum(abs(w) for w in ty_trong)
    #         print(f"⚠️ Cảnh báo: Tổng mức đòn bẩy (Gross Exposure) đang là {tong_leverage * 100:.1f}% Vốn.")
    #         break
    #     except ValueError:
    #         print("❌ Lỗi định dạng! Vui lòng chỉ nhập số và dấu phẩy.")
    # 2. Đưa file dữ liệu vào cỗ máy
    von_dau_tu = 10000000000.0  
    CI = 0.99  
    # Thứ tự: EURUSD, VANG, HPG, VNM, TCB
    ty_trong = [0.2, 0.4, 0.15, 0.1, 0.2]
    tong_leverage = sum(abs(w) for w in ty_trong)
    print(f"✅ Vốn khởi tạo: {von_dau_tu:,.0f} VND")
    print(f"✅ Độ tin cậy VaR/ES: {CI*100}%")
    print(f"✅ Tỷ trọng danh mục: {ty_trong}")
    print(f"⚠️ Cảnh báo: Tổng mức đòn bẩy (Gross Exposure) đang là {tong_leverage * 100:.1f}% Vốn.")
    engine = MarketRisk(df_gia, weights_base=ty_trong, von_ban_dau=von_dau_tu)

    so_hop_dong = engine.get_hedge_contracts()
    if engine.w_f1m < 0:
        print(f"  🔥 YÊU CẦU TRADER: BÁN KHỐNG (SHORT) {so_hop_dong} HỢP ĐỒNG VN30F1M")
    elif engine.w_f1m > 0:
        print(f"  🔥 YÊU CẦU TRADER: MUA (LONG) {so_hop_dong} HỢP ĐỒNG VN30F1M")
    else:
        print(f"  🔥 YÊU CẦU TRADER: Không cần phòng vệ phái sinh.")
    # 3. In Báo cáo chuẩn Basel
    print("="*75)
    print(" BÁO CÁO QUẢN TRỊ RỦI RO DANH MỤC ĐA TÀI SẢN ".center(75))
    print("="*75)

    
    print(f"\n[1] CÁC CHỈ SỐ ĐO LƯỜNG RỦI RO (ĐỘ TIN CẬY {CI*100:.0f}%):")
    print(f"  - Historical VaR:       {engine.historical_var(CI):,.0f} VND")
    print(f"  - Parametric VaR:       {engine.parametric_var(CI):,.0f} VND")
    print(f"  - Cornish-Fisher VaR:   {engine.cornish_fisher_var(CI):,.0f} VND ")
    print(f"  - Expected Shortfall:   {engine.expected_shortfall(CI):,.0f} VND (Mức lỗ phần đuôi)")
    # FRTB mặc định dùng 97.5% theo luật Basel
    print(f"  - FRTB 10days ES:       {engine.frtb_10day_expected_shortfall():,.0f} VND (Chuẩn khung FRTB - Basel III/IV)")
    print(f"  - Stressed ES:          {engine.stressed_expected_shortfall():,.0f} VND (Chuẩn khung FRTB - Basel III/IV)")

    # 4. In kết quả Backtesting
    so_lan_vi_pham, tong_so_ngay_test, df_history = engine.backtest(CI, window=250)
    so_lan_cho_phep = int(tong_so_ngay_test * (1 - CI))
    
    print(f"\n[2] KIỂM ĐỊNH MÔ HÌNH (OUT-OF-SAMPLE BACKTESTING) - TỔNG {tong_so_ngay_test} NGÀY:")
    print(f"  - Cửa sổ trượt (Rolling Window): 250 ngày")
    print(f"  - Số ngày vi phạm thực tế: {so_lan_vi_pham} ngày")
    print(f"  - Số ngày cho phép lý thuyết ({(1-CI)*100:.0f}%): ~{so_lan_cho_phep} ngày")
    
    print("\n-> KẾT LUẬN TRAFFIC LIGHT (ĐÈN GIAO THÔNG BASEL):")
    if so_lan_vi_pham <= so_lan_cho_phep:
        print("   🟢 MÔ HÌNH XANH LÁ: Hệ thống Out-of-sample hoạt động cực kỳ chính xác.")
    elif so_lan_vi_pham <= so_lan_cho_phep + 4: 
        print("   🟡 MÔ HÌNH VÙNG VÀNG: Rủi ro đang tăng cao, cần theo dõi thêm.")
    else:
        print("   🔴 MÔ HÌNH BÁO ĐỘNG: Hệ thống đánh giá quá thấp rủi ro thực tế. Cần hiệu chỉnh lại!")
    engine.check_concentration_risk(hhi_threshold=0.2, single_asset_limit=0.25)

    print("="*75)

⏳ Đang cắm ống hút dữ liệu vào BigQuery...


C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Đã kéo thành công dữ liệu từ BigQuery!
               HỆ THỐNG KHỞI TẠO THÔNG SỐ RỦI RO (DESK INPUT)              
✅ Vốn khởi tạo: 10,000,000,000 VND
✅ Độ tin cậy VaR/ES: 99.0%
✅ Tỷ trọng danh mục: [0.2, 0.4, 0.15, 0.1, 0.2]
⚠️ Cảnh báo: Tổng mức đòn bẩy (Gross Exposure) đang là 105.0% Vốn.
  🔥 YÊU CẦU TRADER: BÁN KHỐNG (SHORT) 20 HỢP ĐỒNG VN30F1M
                BÁO CÁO QUẢN TRỊ RỦI RO DANH MỤC ĐA TÀI SẢN                

[1] CÁC CHỈ SỐ ĐO LƯỜNG RỦI RO (ĐỘ TIN CẬY 99%):
  - Historical VaR:       -178,901,404 VND
  - Parametric VaR:       -153,746,120 VND
  - Cornish-Fisher VaR:   -576,934,656 VND 
  - Expected Shortfall:   -281,336,305 VND (Mức lỗ phần đuôi)
  - FRTB 10days ES:       -566,198,910 VND (Chuẩn khung FRTB - Basel III/IV)
  - Stressed ES:          -272,893,304 VND (Chuẩn khung FRTB - Basel III/IV)

[2] KIỂM ĐỊNH MÔ HÌNH (OUT-OF-SAMPLE BACKTESTING) - TỔNG 1832 NGÀY:
  - Cửa sổ trượt (Rolling Window): 250 ngày
  - Số ngày vi phạm thực tế: 32 ngày
  - Số ngày cho phép lý th

In [2]:
# ====================================================================
# PHẦN 3: THE RETURN TRIP - LƯU BÁO CÁO RỦI RO ĐA TÀI SẢN LÊN BIGQUERY
# ====================================================================
print("\n-> Bắt đầu đóng gói Báo cáo Market Risk (Basel III )...")

from datetime import datetime
from google.cloud import bigquery
from google.oauth2 import service_account

# 1. Gom các chỉ số Toán học vào 1 dòng dữ liệu (Dạng Dictionary)
data_market_risk = {
    "Report_Date": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    "Capital_VND": [von_dau_tu],
    "Confidence_Level": [CI],
    "Gross_Exposure_Pct": [tong_leverage],
    "Historical_VaR": [engine.historical_var(CI)],
    "Parametric_VaR": [engine.parametric_var(CI)],
    "Cornish_Fisher_VaR": [engine.cornish_fisher_var(CI)],
    "Expected_Shortfall": [engine.expected_shortfall(CI)],
    "FRTB_10Day_ES": [engine.frtb_10day_expected_shortfall()],
    "Stressed_ES": [engine.stressed_expected_shortfall()],
    "OOS_Backtest_Violations": [so_lan_vi_pham], 
    "HHI_Concentration_Index": [np.sum(engine.weights_full ** 2)]
}

import pandas as pd
df_market_risk_report = pd.DataFrame(data_market_risk)

# 2. Cấu hình BigQuery
project_id = "jda-k1"
dataset_id = "cuongnm_data_pipeline"
table_report_id = f"{project_id}.{dataset_id}.Report_Market_Risk_Daily"
key_path = r"C:\Users\DELL\Desktop\Project\Project Market risk\Key - path 2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

# 3. Kỹ thuật APPEND: Nối tiếp thêm 1 dòng mỗi ngày để tạo Trendline lịch sử
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    autodetect=True
)

print(f"⏳ Đang bắn báo cáo lên Big Query (Bảng {table_report_id})...")
client.load_table_from_dataframe(df_market_risk_report, table_report_id, job_config=job_config).result()
print(f"✅ XUẤT SẮC! Báo cáo Market Risk đã được lưu trữ an toàn trên BigQuery!")

# ====================================================================
# ĐẨY BẢNG LỊCH SỬ BACKTEST LÊN BIGQUERY (DÙNG ĐỂ VẼ BIỂU ĐỒ)
# ====================================================================
# Format cột Date sang chuỗi để tránh lỗi BigQuery
df_history['Date'] = pd.to_datetime(df_history['Date']).astype(str)

table_history_id = f"{project_id}.{dataset_id}.Report_Market_Risk_Backtest"

# Dùng WRITE_TRUNCATE vì bảng lịch sử này đã được cỗ máy tự sinh ra đủ 1831 ngày
job_config_history = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True
)

print(f"⏳ Đang bắn Lịch sử Backtest lên Đám mây (Bảng {table_history_id})...")
client.load_table_from_dataframe(df_history, table_history_id, job_config=job_config_history).result()
print(f"✅ XUẤT SẮC! Cả Báo cáo Snapshot (Daily) và Lịch sử Backtest (Time-series) đã sẵn sàng trên BigQuery!")



-> Bắt đầu đóng gói Báo cáo Market Risk (Basel III )...
⏳ Đang bắn báo cáo lên Big Query (Bảng jda-k1.cuongnm_data_pipeline.Report_Market_Risk_Daily)...
✅ XUẤT SẮC! Báo cáo Market Risk đã được lưu trữ an toàn trên BigQuery!
⏳ Đang bắn Lịch sử Backtest lên Đám mây (Bảng jda-k1.cuongnm_data_pipeline.Report_Market_Risk_Backtest)...
✅ XUẤT SẮC! Cả Báo cáo Snapshot (Daily) và Lịch sử Backtest (Time-series) đã sẵn sàng trên BigQuery!
